In [1]:
import sys
sys.path.append('/home/pascalgrosset/projects/dsi/examples/halos/mini-dsi')
from store import Store

In [19]:
import random
import string
import os
import csv
import json
from pathlib import Path

In [3]:
cosmology_db = Store('test-cosmo-db-15-01')

In [4]:
files = """
FSN_0.5161_VEL_380.189_TEXP_8.597_BETA_0.5806_SEED_1.342e6
FSN_0.2452_VEL_341.979_TEXP_9.468_BETA_0.2323_SEED_1.071e6
FSN_0.8774_VEL_165.577_TEXP_8.790_BETA_0.2903_SEED_1.303e6
FSN_0.5613_VEL_214.289_TEXP_8.887_BETA_0.0290_SEED_1.845e6
FSN_0.5387_VEL_149.279_TEXP_9.613_BETA_0.8710_SEED_9.548e5
FSN_0.6742_VEL_325.087_TEXP_8.839_BETA_0.4935_SEED_1.613e6
FSN_0.7645_VEL_103.753_TEXP_8.645_BETA_0.8419_SEED_1.535e6
FSN_0.3806_VEL_115.080_TEXP_8.935_BETA_0.1452_SEED_8.774e5
FSN_0.5839_VEL_84.333_TEXP_9.903_BETA_0.3194_SEED_1.458e6
FSN_0.2677_VEL_157.036_TEXP_8.984_BETA_0.4355_SEED_1.961e6
FSN_0.8323_VEL_203.704_TEXP_9.516_BETA_0.4645_SEED_8.387e5
FSN_0.4935_VEL_293.089_TEXP_9.081_BETA_0.2613_SEED_8e5
FSN_0.4258_VEL_98.401_TEXP_9.661_BETA_0.4065_SEED_1.187e6
FSN_0.7419_VEL_399.945_TEXP_9.177_BETA_0.3774_SEED_9.935e5
FSN_0.2000_VEL_121.060_TEXP_9.226_BETA_0.7839_SEED_1.226e6
FSN_0.2903_VEL_250.611_TEXP_8.694_BETA_0.6968_SEED_9.161e5
FSN_0.6968_VEL_141.579_TEXP_9.371_BETA_0_SEED_1.381e6
FSN_0.6290_VEL_360.579_TEXP_9.758_BETA_0.8129_SEED_1.768e6
FSN_0.4032_VEL_225.944_TEXP_9.129_BETA_0.9000_SEED_1.419e6
FSN_0.6516_VEL_183.654_TEXP_8.548_BETA_0.1742_SEED_1.148e6
FSN_0.2226_VEL_193.197_TEXP_9.710_BETA_0.6097_SEED_1.497e6
FSN_0.4710_VEL_88.716_TEXP_8.742_BETA_0.0871_SEED_1.032e6
FSN_0.8097_VEL_127.644_TEXP_9.806_BETA_0.2032_SEED_1.884e6
FSN_0.4484_VEL_174.181_TEXP_9.952_BETA_0.7258_SEED_1.652e6
FSN_0.7871_VEL_238.232_TEXP_10_BETA_0.5516_SEED_1.110e6
FSN_0.9000_VEL_264.241_TEXP_9.274_BETA_0.1161_SEED_1.574e6
FSN_0.8548_VEL_93.541_TEXP_9.032_BETA_0.6677_SEED_1.729e6
FSN_0.3581_VEL_79.983_TEXP_9.323_BETA_0.5226_SEED_1.806e6
FSN_0.6065_VEL_109.144_TEXP_9.419_BETA_0.6387_SEED_2e6
FSN_0.7194_VEL_277.971_TEXP_9.565_BETA_0.7548_SEED_1.923e6
FSN_0.3355_VEL_308.319_TEXP_9.855_BETA_0.0581_SEED_1.265e6
FSN_0.3129_VEL_134.586_TEXP_8.500_BETA_0.3484_SEED_1.690e6
"""
files_and_dirs = files.strip().split('\n')

In [42]:
def parse_params(file_path):
    if not os.path.exists(file_path):
        print(f"File '{file_path}' does not exist")
        return
    
    key_value_map = {}

    with open(file_path, 'r') as file:
        for line in file:
            # Skip lines that start with '#'
            if line.strip().startswith('#'):
                continue

            if line.strip() == "":
                continue
    
            try:
                key, value = line.split(' ', 1)
            except ValueError:
                key = line.split(' ', 1)[0]
                value = ""
                print(f"Warning: line: {line}")

            key = key.strip()
            value = value.strip()
            key_value_map[key] = value

    return key_value_map

In [8]:
def parse_folders(prefix, files_and_dirs):
    ## Get the list of files and folders: disabled for now
    # files_and_dirs = os.listdir(folder_path)
    
    # Select only those that start with 
    filtered_items = [item for item in files_and_dirs if item.startswith(prefix)]
    
    # Put the data into a new fomat
    data = []
    header = filtered_items[0].split('_')
    data.append(header[::2])
    
    for f in filtered_items:
        values =  f.split('_')
        data.append(values[1::2])

    return data

In [30]:
def append_col(data, new_column):
    for i in range(len(data)):
        data[i].append(new_column[i])
    return data

In [31]:
def random_string(length):
    letters = string.ascii_letters  # Generates a mix of uppercase and lowercase letters
    return ''.join(random.choice(letters) for _ in range(length))

In [36]:
data = parse_folders('FSN', files_and_dirs)

In [38]:
num_entries = len(data)

location_col = ['path']
for i in range(num_entries-1):
    location_col.append('/lus/eagle/projects/CosDiscover/nfrontiere/SCIDAC_RUNS/128MPC_RUNS_FLAMINGO_DESIGN_3B/')

id_col = ['sim_id']
for i in range(num_entries-1):
    id_col.append(random_string(5))

run_col = ['run_id']
for i in range(num_entries-1):
    run_col.append('B')

In [39]:
data = append_col(data, location_col)
data = append_col(data, run_col)
data = append_col(data, id_col)

In [41]:
cosmology_db.ingest_table(data, 'runs')

Table 'runs' created.
Number of rows inserted: '33'.


In [43]:
indat_params = parse_params('/home/pascalgrosset/Desktop/indat.params')

In [45]:
cosmotools_params = parse_params('/home/pascalgrosset/Desktop/cosmotools-config.dat')

In [46]:
cosmotools_params

{'VERSION': '1.0',
 'VISUALIZATION': 'NO',
 'VIZ_SERVER': '127.0.0.1',
 'VIZ_PORT': '2222',
 'VIZ_FREQUENCY': '20',
 'CONFIG_REFRESH_FREQUENCY': '-1',
 'ANALYSISTOOL': 'CHECKPOINT_IO  NO',
 'SECTION': 'HALOFINDER',
 'INSTANCE_NAME': 'LANLHALOFINDER',
 'FREQUENCY_TYPE': '0',
 'EXPLICIT_TIMESTEPS': '53 54 55 57 58 59 60 62 63 65 66 67 68 70 71 74 75 78 79 81 84 85 86 88 90 93 95 97 99 101 105 108 110 113 115 119 122 125 128 132 134 138 142 145 149 151 153 155 159 164 168 171 175 176 180 185 189 194 199 204 205 209 214 220 224 225 230 237 243 247 248 254 260 267 274 275 280 288 294 301 308 310 316 325 334 341 349 355 357 368 374 384 393 404 415 423 435 445 458 468 479 490 498 502 515 528 542 552 567 583 594 612 624',
 'IMPLICIT_TIMESTEPS': '1',
 'WRITE_OUTPUT': 'YES',
 'BASE_OUTPUT_FILE_NAME': './analysis/m000p',
 'VISIBLE': 'YES',
 'LINKING_LENGTH': '0.168 # The linking length used for FOF algorithm',
 'MINIMUM_MASS': '1.0e5 # The minimum mass (M_solar/h)',
 'FOF_PMIN': '20    # Minimum 